In [2]:
!pip install nba_api

  Using cached nba_api-1.11.4-py3-none-any.whl.metadata (5.8 kB)
Using cached nba_api-1.11.4-py3-none-any.whl (322 kB)


In [2]:
# Loading player bio data
import pandas as pd
import time
from pathlib import Path
from nba_api.stats.endpoints import LeagueDashPlayerBioStats
def make_NBA_season_string(end_year):
    return f"{end_year - 1}-{str(end_year)[-2:]}"
all_bio_stats = []
# Downloading all seasons
for end_year in range(1997, 2027):
    season = make_NBA_season_string(end_year)
    print("Loaded:", season)
    try:
        bio = LeagueDashPlayerBioStats(
            season=season,
            season_type_all_star="Regular Season",
            per_mode_simple="Totals",
            timeout=60
        ).get_data_frames()[0]
        bio["Season"] = season
        bio["Year"] = end_year
        all_bio_stats.append(bio)
        time.sleep(2)
    except Exception as e:
        print("Could not load:", season, e)
# Combining all the seasons from 1997 to 2026 into bio_stats_all
if all_bio_stats:
    bio_stats_all = pd.concat(all_bio_stats, ignore_index=True)
# Saving raw data into Local Directory
MyPath = Path.home() / "Desktop" / "NBA_Project" / "data" / "raw"
MyPath.mkdir(parents=True, exist_ok=True)
The_output_file = MyPath / "bio_stats_all_raw.csv"
bio_stats_all.to_csv(The_output_file, index=False)
print("Saved to:", The_output_file)
print("Shape:", bio_stats_all.shape)
bio_stats_all.head()

Loaded: 1996-97
Loaded: 1997-98
Loaded: 1998-99
Loaded: 1999-00
Loaded: 2000-01
Loaded: 2001-02
Loaded: 2002-03
Loaded: 2003-04
Loaded: 2004-05
Loaded: 2005-06
Loaded: 2006-07
Loaded: 2007-08
Loaded: 2008-09
Loaded: 2009-10
Loaded: 2010-11
Loaded: 2011-12
Loaded: 2012-13
Loaded: 2013-14
Loaded: 2014-15
Loaded: 2015-16
Loaded: 2016-17
Loaded: 2017-18
Loaded: 2018-19
Loaded: 2019-20
Loaded: 2020-21
Loaded: 2021-22
Loaded: 2022-23
Loaded: 2023-24
Loaded: 2024-25
Loaded: 2025-26
Saved to: C:\Users\ahmed\Desktop\NBA_Project\data\raw\bio_stats_all_raw.csv
Shape: (14569, 25)


,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,AGE,PLAYER_HEIGHT,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,COLLEGE,COUNTRY,...,REB,AST,NET_RATING,OREB_PCT,DREB_PCT,USG_PCT,TS_PCT,AST_PCT,Season,Year
0,920,A.C. Green,1610612742,DAL,33.0,6-9,81.0,225,Oregon State,USA,...,656,69,-7.4,0.089,0.184,0.118,0.523,0.045,1996-97,1997
1,243,Aaron McKie,1610612765,DET,24.0,6-5,77.0,209,Temple,USA,...,221,161,3.7,0.026,0.113,0.142,0.524,0.163,1996-97,1997
2,1425,Aaron Williams,1610612763,VAN,25.0,6-9,81.0,225,Xavier,USA,...,143,15,-9.3,0.113,0.144,0.161,0.599,0.051,1996-97,1997
3,768,Acie Earl,1610612749,MIL,27.0,6-11,83.0,240,Iowa,USA,...,96,20,-6.4,0.067,0.122,0.220,0.435,0.077,1996-97,1997
4,228,Adam Keefe,1610612762,UTA,27.0,6-9,81.0,241,Stanford,USA,...,216,32,7.2,0.096,0.158,0.124,0.572,0.051,1996-97,1997


In [ ]:
########################## clean player bio data #################################################################

In [4]:
# Selecting the useful bio and performance variables
bio_info = bio_stats_all[[
    "PLAYER_ID","PLAYER_NAME","TEAM_ID","TEAM_ABBREVIATION","Season","Year",
    "AGE","PLAYER_HEIGHT_INCHES","PLAYER_WEIGHT","COLLEGE","COUNTRY",
    "DRAFT_YEAR","DRAFT_ROUND","DRAFT_NUMBER","GP","PTS","REB","AST",
    "NET_RATING","OREB_PCT","DREB_PCT","USG_PCT","TS_PCT","AST_PCT"
]].copy()
# Converting variables to numeric columns 
bio_info["PLAYER_HEIGHT_INCHES"] = pd.to_numeric(bio_info["PLAYER_HEIGHT_INCHES"], errors="coerce")
bio_info["PLAYER_WEIGHT"] = pd.to_numeric(bio_info["PLAYER_WEIGHT"], errors="coerce")
bio_info["AGE"] = pd.to_numeric(bio_info["AGE"], errors="coerce")
# Converting height and weight to cm and kg
bio_info["height_cm"] = (bio_info["PLAYER_HEIGHT_INCHES"] * 2.54).round(1)
bio_info["weight_kg"] = (bio_info["PLAYER_WEIGHT"] * 0.453592).round(1)
# Renaming the columns into more readable way
bio_info = bio_info.rename(columns={
    "PLAYER_ID":"player_id","PLAYER_NAME":"player_name","TEAM_ID":"team_id",
    "TEAM_ABBREVIATION":"team","Season":"season","Year":"year","AGE":"age",
    "COLLEGE":"college","COUNTRY":"country","DRAFT_YEAR":"draft_year",
    "DRAFT_ROUND":"draft_round","DRAFT_NUMBER":"draft_number",
    "GP":"games_played","PTS":"points","REB":"rebounds","AST":"assists",
    "NET_RATING":"net_rating","OREB_PCT":"offensive_rebound_pct",
    "DREB_PCT":"defensive_rebound_pct","USG_PCT":"usage_pct",
    "TS_PCT":"true_shooting_pct","AST_PCT":"assist_pct"
})
# Cleaning age format
bio_info["age"] = bio_info["age"].astype("Int64")
# Reordering columns
bio_info = bio_info[[
    "year","season","player_id","player_name","team_id","team","age",
    "height_cm","weight_kg","college","country","draft_year","draft_round",
    "draft_number","games_played","points","rebounds","assists","net_rating",
    "usage_pct","true_shooting_pct","assist_pct","offensive_rebound_pct",
    "defensive_rebound_pct"
]]
print("Original bio_stats_all shape:", bio_stats_all.shape)
print("Clean bio_info shape:", bio_info.shape)
bio_info.head(50)
bio_info.columns.tolist()

Original bio_stats_all shape: (14569, 25)
Clean bio_info shape: (14569, 24)


['year',
 'season',
 'player_id',
 'player_name',
 'team_id',
 'team',
 'age',
 'height_cm',
 'weight_kg',
 'college',
 'country',
 'draft_year',
 'draft_round',
 'draft_number',
 'games_played',
 'points',
 'rebounds',
 'assists',
 'net_rating',
 'usage_pct',
 'true_shooting_pct',
 'assist_pct',
 'offensive_rebound_pct',
 'defensive_rebound_pct']

In [13]:
################################################ Manual Correction of Data #######################################

In [5]:
# Checking the missing height and weight
print("Missing height before:", bio_info["height_cm"].isna().sum())
print("Missing weight before:", bio_info["weight_kg"].isna().sum())

# Filling values from other seasons of the same player
bio_info = bio_info.sort_values(["player_id","year"]).copy()
bio_info["height_cm"] = bio_info.groupby("player_id")["height_cm"].transform(lambda x: x.ffill().bfill())
bio_info["weight_kg"] = bio_info.groupby("player_id")["weight_kg"].transform(lambda x: x.ffill().bfill())
print("Missing height after:", bio_info["height_cm"].isna().sum())
print("Missing weight after:", bio_info["weight_kg"].isna().sum())

# Checking remaining missing height and weight players after the filling from other seasons

missing_height_weight_players = bio_info[
    bio_info["height_cm"].isna() | bio_info["weight_kg"].isna()
][
    ["year", "season", "player_id", "player_name", "team", "height_cm", "weight_kg", "age"]
].sort_values(
    ["player_name", "year"]
)
missing_height_weight_players

Missing height before: 71
Missing weight before: 79
Missing height after: 10
Missing weight after: 18


,year,season,player_id,player_name,team,height_cm,weight_kg,age
14080,2026,2025-26,1642959,Chris Youngblood,POR,193.0,NaN,24
2380,2002,2001-02,1829,Ike Fontaine,MEM,NaN,NaN,27
14214,2026,2025-26,1643141,Jahmyl Telfort,LAC,200.7,NaN,25
1957,2001,2000-01,1554,Jamal Robinson,MIA,NaN,NaN,27
13678,2025,2024-25,1642377,Jaylen Wells,MEM,203.2,NaN,21
14249,2026,2025-26,1642377,Jaylen Wells,MEM,200.7,NaN,22
8337,2015,2014-15,203968,Jerrelle Benimon,DEN,NaN,NaN,23
2006,2001,2000-01,2064,Khalid El-Amin,CHI,NaN,NaN,22
14344,2026,2025-26,1643018,LJ Cryer,GSW,182.9,NaN,24
14350,2026,2025-26,1643133,Lawson Lovering,MEM,213.4,NaN,23


In [ ]:
# Showing original draft information for the three players
# here the old Columns are used to show old values
draft_player_ids = [202536,201041,2891]
bio_stats_all.loc[
    bio_stats_all["PLAYER_ID"].isin(draft_player_ids),
    ["PLAYER_ID","PLAYER_NAME","DRAFT_YEAR","DRAFT_ROUND","DRAFT_NUMBER"]
].drop_duplicates().sort_values("PLAYER_ID")

,PLAYER_ID,PLAYER_NAME,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER
3804,2891,Mark Jones,1983,4,82
7146,201041,Walker Russell,1982,4,78
6914,202536,Jerry Smith,1963,2,12


In [ ]:
# Correcting wrong draft information
bio_info["draft_year"] = bio_info["draft_year"].astype("object")
bio_info["draft_round"] = bio_info["draft_round"].astype("object")
bio_info["draft_number"] = bio_info["draft_number"].astype("object")
bio_info.loc[bio_info["player_id"] == 202536, ["draft_year","draft_round","draft_number"]] = [2010,"Undrafted","Undrafted"]
bio_info.loc[bio_info["player_id"] == 201041, ["draft_year","draft_round","draft_number"]] = [2006,"Undrafted","Undrafted"]
bio_info.loc[bio_info["player_id"] == 2891, ["draft_year","draft_round","draft_number"]] = [1998,"Undrafted","Undrafted"]

# Adding manual  height and weight corrections based on the references cited on the Readme file.
manual_height_weight_updates = [
    (2237,"Ratko Varda",216,117),
    (203968,"Jerrelle Benimon",203,111),
    (2064,"Khalid El-Amin",178,90),
    (2092,"Ruben Garces",206,111),
    (2121,"Terrance Roberson",201,97),
    (203805,"Will Cherry",183,83),
    (1642959,"Chris Youngblood",193,100),
    (1643141,"Jahmyl Telfort",201,102),
    (1642377,"Jaylen Wells",201,93),
    (1643018,"LJ Cryer",185,91),
    (1643133,"Lawson Lovering",213,111),
    (1642449,"Tolu Smith",211,115),
    (203565,"Patrick Christopher",196,94),
    (1554,"Jamal Robinson",201,96),
    (1829,"Ike Fontaine",193,95),
    (199,"Reggie Williams",201,86)
]
# Applying those manual corrections

for player_id, player_name, height_cm, weight_kg in manual_height_weight_updates:
    mask = bio_info["player_id"] == player_id
    bio_info.loc[mask, "height_cm"] = height_cm
    bio_info.loc[mask, "weight_kg"] = weight_kg
# Checking the number of remaining missing values 
print("Remaining missing height:", bio_info["height_cm"].isna().sum())
print("Remaining missing weight:", bio_info["weight_kg"].isna().sum())

# Saving cleaned bio data localy
toPath = Path.home() / "Desktop" / "NBA_Project" / "data" / "processed"
toPath.mkdir(parents=True, exist_ok=True)
output_file = toPath / "bio_info_clean.csv"
bio_info.to_csv(output_file, index=False)
print("Saved cleaned bio data to:", output_file)
bio_info

Remaining missing height: 0
Remaining missing weight: 0
Saved cleaned bio data to: C:\Users\ahmed\Desktop\NBA_Project\data\processed\bio_info_clean.csv


,year,season,player_id,player_name,team_id,team,age,height_cm,weight_kg,college,...,games_played,points,rebounds,assists,net_rating,usage_pct,true_shooting_pct,assist_pct,offensive_rebound_pct,defensive_rebound_pct
57,1997,1996-97,2,Byron Scott,1610612747,LAL,36,193.0,93.0,Arizona State,...,79,526,118,99,5.9,0.150,0.590,0.110,0.015,0.065
181,1997,1996-97,3,Grant Long,1610612765,DET,31,205.7,112.5,Eastern Michigan,...,65,326,222,39,4.2,0.151,0.523,0.058,0.084,0.129
622,1998,1997-98,3,Grant Long,1610612765,DET,32,205.7,112.5,Eastern Michigan,...,40,141,150,25,-0.4,0.103,0.496,0.056,0.078,0.133
1059,1999,1998-99,3,Grant Long,1610612737,ATL,33,205.7,112.5,Eastern Michigan,...,50,489,296,53,7.9,0.184,0.529,0.078,0.074,0.142
1497,2000,1999-00,3,Grant Long,1610612763,VAN,34,205.7,112.5,Eastern Michigan,...,42,203,234,43,-2.4,0.120,0.512,0.076,0.099,0.173
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14350,2026,2025-26,1643133,Lawson Lovering,1610612763,MEM,23,213.0,111.0,Utah,...,2,14,15,3,-0.1,0.106,0.574,0.070,0.105,0.188
14214,2026,2025-26,1643141,Jahmyl Telfort,1610612746,LAC,25,201.0,102.0,None,...,8,1,3,1,-8.9,0.093,0.087,0.050,0.000,0.088
14010,2026,2025-26,1643158,Andersson Garcia,1610612762,UTA,25,200.7,99.8,Texas A&M,...,5,26,42,14,-9.8,0.094,0.394,0.107,0.089,0.150
14513,2026,2025-26,1643253,Toby Okani,1610612763,MEM,24,203.2,95.3,West Virginia,...,6,60,21,6,-28.1,0.142,0.408,0.039,0.034,0.053


In [ ]:
# Checking remaining missing height and weight players after corrections.
players_to_check = [
    (1642959,"Chris Youngblood"),
    (1829,"Ike Fontaine"),
    (1643141,"Jahmyl Telfort"),
    (1554,"Jamal Robinson"),
    (1642377,"Jaylen Wells"),
    (203968,"Jerrelle Benimon"),
    (2064,"Khalid El-Amin"),
    (1643018,"LJ Cryer"),
    (1643133,"Lawson Lovering"),
    (203565,"Patrick Christopher"),
    (2237,"Ratko Varda"),
    (2092,"Ruben Garces"),
    (2121,"Terrance Roberson"),
    (1642449,"Tolu Smith"),
    (203805,"Will Cherry"),
    (199,"Reggie Williams")
]
manual_original_check = pd.concat([
    bio_info[
        (bio_info["player_id"] == player_id) &
        (bio_info["player_name"] == player_name)
    ][["year","season","player_id","player_name","team","height_cm","weight_kg","age"]].sort_values("year")
    for player_id, player_name in players_to_check
])
manual_original_check

,year,season,player_id,player_name,team,height_cm,weight_kg,age
14080,2026,2025-26,1642959,Chris Youngblood,POR,193.0,100.0,24
2380,2002,2001-02,1829,Ike Fontaine,MEM,193.0,95.0,27
14214,2026,2025-26,1643141,Jahmyl Telfort,LAC,201.0,102.0,25
1957,2001,2000-01,1554,Jamal Robinson,MIA,201.0,96.0,27
13678,2025,2024-25,1642377,Jaylen Wells,MEM,201.0,93.0,21
14249,2026,2025-26,1642377,Jaylen Wells,MEM,201.0,93.0,22
8337,2015,2014-15,203968,Jerrelle Benimon,DEN,203.0,111.0,23
2006,2001,2000-01,2064,Khalid El-Amin,CHI,178.0,90.0,22
14344,2026,2025-26,1643018,LJ Cryer,GSW,185.0,91.0,24
14350,2026,2025-26,1643133,Lawson Lovering,MEM,213.0,111.0,23


In [ ]:
# Showing draft information after the corrections.
draft_player_ids = [202536,201041,2891]
bio_info.loc[
    bio_info["player_id"].isin(draft_player_ids),
    ["player_id","player_name","draft_year","draft_round","draft_number"]
].drop_duplicates().sort_values("player_id")

,player_id,player_name,draft_year,draft_round,draft_number
3804,2891,Mark Jones,1998,Undrafted,Undrafted
7146,201041,Walker Russell,2006,Undrafted,Undrafted
6914,202536,Jerry Smith,2010,Undrafted,Undrafted
